In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline, griddata
from scipy.stats import norm
from skew.utils import *
from skew.data_store import save_option_snapshot, load_option_snapshots, list_snapshots
import nest_asyncio, asyncio
nest_asyncio.apply()


# ===============================
# CONFIG
# ===============================
DATA_SOURCE     = "yfinance"              # "yfinance" or "ibkr"
TICKERS         = ['NVDA', 'QQQ', 'MSFT', 'TSM','CL', 'SPY', 'AAPL', 'AMZN', 'META', 'GOOGL']
MAX_EXPIRIES    = 50
MIN_EXPIRY_DAYS = 10
MAX_EXPIRY_DAYS = 365                     # ignore options expiring beyond 1 year
DB_PATH         = "../option-risk/data/options.db"    # single SQLite file for all snapshots

In [8]:
from skew.zero_curve import RiskFreeCurveFactory
from datetime import date, timedelta

# Create a piecewise linear zero curve for USD (fetches latest from Fed)
curve_usd = RiskFreeCurveFactory.create("USD")

print("Valuation date:", curve_usd.valuation_date)
print("SOFR rate:", round(curve_usd.rate * 100, 3), "%")
print("1Y DF:", round(curve_usd.discount_factor(curve_usd.valuation_date + timedelta(days=360)), 5))



Valuation date: 2026-05-29
SOFR rate: 3.91 %
1Y DF: 0.96217


In [9]:
import pandas as pd
from datetime import date
from skew.zero_curve import PiecewiseLinearZeroCurve

# Usage
pillars = [
    ("1D", 0.04),
    ("1M", 0.0401),
    ("3M", 0.0389),
    ("1Y", 0.0354),
    ("2Y", 0.0333),
    ("5Y", 0.0334),
]
curve = PiecewiseLinearZeroCurve(
    valuation_date=date.today(),
    pillars=pillars,
    day_count="ACT/365F"
)

print("Valuation date:", curve.valuation_date)
print("1D rate:", round(curve.zero_rate(curve.valuation_date + timedelta(days=1))*100, 3), "%")
print("1Y rate:", round(curve.zero_rate(curve.valuation_date + timedelta(days=65))*100, 3), "%")
print("1Y DF:", round(curve.discount_factor(curve.valuation_date + timedelta(days=65)), 5))

# Now you can get discount factors, forwards etc.


Valuation date: 2026-05-29
1D rate: 4.0 %
1Y rate: 3.94 %
1Y DF: 0.99301


In [ ]:
# ===============================
# DATA: yfinance option surface
# ===============================
def fetch_surface_yfinance(ticker, max_expiries=10, min_expiry_days=10, max_expiry_days=365):
    import yfinance as yf

    yf_t = yf.Ticker(ticker)
    spot = yf_t.history(period="1d")["Close"].iloc[-1]

    expiries = yf_t.options
    if not expiries:
        raise ValueError("No expiries returned by yfinance for this ticker.")

    # Filter to within [min_expiry_days, max_expiry_days] before fetching
    today = pd.Timestamp.today().normalize()
    expiries = [
        e for e in expiries
        if min_expiry_days < (pd.to_datetime(e) - today).days <= max_expiry_days
    ][:max_expiries]

    frames = []
    for exp in expiries:
        chain = yf_t.option_chain(exp)
        for opt_type, tbl in [("call", chain.calls), ("put", chain.puts)]:
            tbl = tbl.rename(columns={"impliedVolatility": "iv"})
            tbl["expiry"] = pd.to_datetime(exp).tz_localize(None)
            tbl["isCall"] = (opt_type == "call")
            cols = ["strike", "iv", "lastPrice", "bid", "ask",
                    "openInterest", "volume", "expiry", "inTheMoney", "isCall"]
            frames.append(tbl[[c for c in cols if c in tbl.columns]])

    df = pd.concat(frames, ignore_index=True)

    # ── Liquidity filters ─────────────────────────────────────────────────────
    if "bid" in df.columns:
        df = df[df["bid"] > 0]
    df = df[df["strike"].between(spot * 0.40, spot * 2.50)]
    if "openInterest" in df.columns:
        df = df[df["openInterest"].fillna(0) > 0]
    df = df[df["iv"].between(0.02, 2.50)].copy()

    # ── Drop columns the DB schema doesn't know about ─────────────────────────
    df = df.drop(columns=["openInterest", "volume", "inTheMoney"], errors="ignore")

    # ── Time filters ──────────────────────────────────────────────────────────
    df["days_to_expiry"] = (df["expiry"] - pd.Timestamp.today().normalize()).dt.days
    df["T"] = df["days_to_expiry"] / 365.0

    # ── Rates / forward ───────────────────────────────────────────────────────
    df["expiry_date"] = df["expiry"].dt.date
    df["disc_factor"] = df["expiry_date"].apply(curve_usd.discount_factor)
    info = yf_t.info or {}
    q_div = info.get("dividendYield", 0.0) or 0.0
    df["div_yield"] = float(q_div / 100)
    df["forward"] = spot * np.exp(-q_div / 100 * df["T"]) * df["disc_factor"]
    df["spot"] = spot

    n = save_option_snapshot(df, ticker, db_path=DB_PATH, source="yfinance")
    print(f"  yfinance: saved {n} new rows for {ticker}  (spot={spot:.2f})")

In [ ]:
# ===============================
# DATA: IBKR option surface
# (requires IB Gateway/TWS running locally on port 7497)
# Only used when DATA_SOURCE = "ibkr"
# ===============================
import logging

try:
    from ib_insync import IB, Stock, Option, util
    _IBKR_AVAILABLE = True
except ImportError:
    _IBKR_AVAILABLE = False
    print("ib_insync not installed — IBKR fetch disabled. Run: pip install ib_insync")


async def fetch_surface_ibkr_nb(ticker: str, max_expiries=10, batch_size=10):
    if not _IBKR_AVAILABLE:
        raise RuntimeError("ib_insync is not installed. Run: pip install ib_insync")

    util.startLoop()
    ib = IB()
    util.logToConsole(level=logging.CRITICAL)

    if not ib.isConnected():
        await ib.connectAsync("127.0.0.1", 7497, clientId=1)
        print("Connected:", ib.isConnected())

    ib.reqMarketDataType(4)
    ib.sleep(2.0)
    stk = Stock(ticker, "SMART", "USD")
    ib.qualifyContracts(stk)
    mkt = ib.reqMktData(stk, "", False, False)
    ib.sleep(1.0)
    spot = float(mkt.last or mkt.close)

    chains = ib.reqSecDefOptParams(stk.symbol, "", stk.secType, stk.conId)
    chain = [c for c in chains if c.tradingClass == stk.symbol][0]

    today = pd.Timestamp.today()
    min_date = today + pd.Timedelta(days=7)
    max_date = today + pd.Timedelta(days=365)   # cap at 1 year
    expiries = sorted(
        pd.to_datetime(exp) for exp in chain.expirations
        if min_date <= pd.to_datetime(exp) <= max_date
    )[:max_expiries]

    rows = []
    for exp in expiries:
        exp_str = exp.strftime("%Y%m%d")
        strikes = sorted(k for k in chain.strikes if k % 1 == 0)
        contracts = [
            Option(symbol=ticker, lastTradeDateOrContractMonth=exp_str,
                   strike=k, right=right, exchange="SMART", currency="USD")
            for k in strikes for right in ("C", "P")
        ]
        ib.qualifyContracts(*contracts)

        for i in range(0, len(contracts), batch_size):
            batch = contracts[i:i + batch_size]
            for c in batch:
                ib.reqMktData(c, "", False, False)
            ib.sleep(2.0)
            for c in batch:
                tk = ib.ticker(c)
                if not tk:
                    continue
                g = getattr(tk, "modelGreeks", None)
                delta = getattr(g, "delta", None)
                if not delta:
                    continue
                undPrice = getattr(g, "undPrice", None)
                if ((c.right == "C" and (0.1 < delta < 0.9) and c.strike < undPrice) or
                        (c.right == "P" and (0.1 < -delta < 0.9) and c.strike > undPrice)):
                    iv = getattr(g, "impliedVol", None)
                    if iv and 0.0001 < iv < 3.0:
                        bid = tk.bid or np.nan
                        ask = tk.ask or np.nan
                        mid = (0.5 * (bid + ask) if np.isfinite(bid) and np.isfinite(ask) and bid > 0 and ask > 0
                               else bid if np.isfinite(bid) and bid > 0
                               else ask if np.isfinite(ask) and ask > 0
                               else np.nan)
                        rows.append({
                            "strike": c.strike,
                            "type": c.right,
                            "expiry": pd.to_datetime(c.lastTradeDateOrContractMonth),
                            "delta": float(delta),
                            "spot": float(undPrice),
                            "lastPrice": float(mid),
                            "bid": bid,
                            "ask": ask,
                            "iv": float(iv),
                        })

    ib.disconnect()
    df = pd.DataFrame(rows)
    df["days_to_expiry"] = (df["expiry"] - pd.Timestamp.today().normalize()).dt.days
    df = df[df["days_to_expiry"] > 1].copy()
    df["T"] = df["days_to_expiry"] / 365.0
    # Enrich with discount factor and forward price (same as yfinance path)
    df["expiry_date"] = df["expiry"].dt.date
    df["disc_factor"] = df["expiry_date"].apply(curve_usd.discount_factor)
    df["div_yield"] = 0.0   # IBKR doesn't supply div yield directly; override if known
    df["forward"] = df["spot"] * df["disc_factor"]
    n = save_option_snapshot(df, ticker, db_path=DB_PATH, source="ibkr")
    print(f"  IBKR: saved {n} new rows for {ticker}")

In [ ]:
if __name__ == "__main__":
    for TICKER in TICKERS:
        print(f"Fetching {TICKER} from {DATA_SOURCE}...")
        if DATA_SOURCE == "ibkr":
            asyncio.run(fetch_surface_ibkr_nb(TICKER, max_expiries=MAX_EXPIRIES, batch_size=10))
        elif DATA_SOURCE == "yfinance":
            fetch_surface_yfinance(
                TICKER,
                max_expiries=MAX_EXPIRIES,
                min_expiry_days=MIN_EXPIRY_DAYS,
                max_expiry_days=MAX_EXPIRY_DAYS,
            )
        else:
            raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

    print("\nDB snapshot summary:")
    print(list_snapshots(db_path=DB_PATH).to_string(index=False))